# 7-MLET Datathon: Contextual Bandits for Financial Offers

This notebook builds a complete, reproducible adaptive experimentation pipeline for marketing offers.

What you will get:
- robust dataset loading with local-first logic and OpenML fallback
- preprocessing with explicit leakage handling (`duration` removed)
- synthetic contextual bandit environment with delayed rewards
- deterministic baseline, UCB, and Thompson Sampling policies
- evaluation with cumulative reward, regret, conversion rate, and exploration share

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.datasets import fetch_openml

RNG = np.random.default_rng(42)
pd.set_option("display.max_columns", 120)

## 1) Load Data (Kaggle API First, then Local/OpenML Fallback)

For 7-MLET compliance, we first attempt to load a Kaggle dataset using the official `kaggle` library.

Expected setup before running:
- `pip install kaggle`
- Kaggle credentials available as `~/.kaggle/kaggle.json` (or `KAGGLE_USERNAME` + `KAGGLE_KEY` env vars)

Fallback order used by this notebook:
1. Kaggle API download (`henriqueyamahata/bank-marketing`, then alternatives)
2. Local CSV files in common project paths
3. OpenML `bank-marketing` dataset (reproducibility fallback)

In [3]:
def _try_read_csv(path: Path) -> Optional[pd.DataFrame]:
    if not path.exists() or path.suffix.lower() != ".csv":
        return None
    for sep in [",", ";", "\t", "|"]:
        try:
            df = pd.read_csv(path, sep=sep)
            if df.shape[1] > 2:
                return df
        except Exception:
            continue
    return None


def _find_target_column(columns: Sequence[str]) -> Optional[str]:
    lowered = {c.lower(): c for c in columns}
    candidates = [
        "y",
        "target",
        "label",
        "subscribed",
        "conversion",
        "converted",
        "response",
        "class",
    ]
    for cand in candidates:
        if cand in lowered:
            return lowered[cand]
    return None


def _try_load_from_kaggle(repo_root: Path) -> Optional[Tuple[pd.DataFrame, str, str]]:
    kaggle_dir = repo_root / "tmp" / "7mlet" / "kaggle_bank_marketing"
    kaggle_dir.mkdir(parents=True, exist_ok=True)

    csv_candidates = sorted(kaggle_dir.rglob("*.csv"))

    if not csv_candidates:
        datasets = [
            "henriqueyamahata/bank-marketing",
            "tunguz/bank-marketing-data-set",
            "dharmik34/bank-term-deposit-subscription",
        ]
        try:
            from kaggle.api.kaggle_api_extended import KaggleApi

            api = KaggleApi()
            api.authenticate()

            for ds in datasets:
                try:
                    api.dataset_download_files(ds, path=str(kaggle_dir), unzip=True, quiet=True)
                except Exception:
                    continue

            csv_candidates = sorted(kaggle_dir.rglob("*.csv"))
        except Exception as ex:
            print(f"Kaggle load skipped: {ex}")
            return None

    for path in csv_candidates:
        df = _try_read_csv(path)
        if df is None:
            continue
        target_col = _find_target_column(df.columns)
        if target_col is None:
            continue
        return df, target_col, f"kaggle dataset file: {path.as_posix()}"

    return None


def load_bank_marketing_dataset() -> Tuple[pd.DataFrame, str, str]:
    repo_root = Path.cwd()

    kaggle_loaded = _try_load_from_kaggle(repo_root)
    if kaggle_loaded is not None:
        return kaggle_loaded

    candidates = [
        repo_root / "tmp" / "7mlet" / "bank-additional-full.csv",
        repo_root / "tmp" / "7mlet" / "bank-full.csv",
        repo_root / "tmp" / "7mlet" / "bank.csv",
        repo_root / "data" / "assets" / "bank-additional-full.csv",
        repo_root / "data" / "assets" / "bank-full.csv",
    ]

    for path in candidates:
        df = _try_read_csv(path)
        if df is None:
            continue
        target_col = _find_target_column(df.columns)
        if target_col is None:
            continue
        return df, target_col, f"local CSV: {path.as_posix()}"

    try:
        bunch = fetch_openml(name="bank-marketing", version=1, as_frame=True, parser="auto")
    except TypeError:
        bunch = fetch_openml(name="bank-marketing", version=1, as_frame=True)

    df = bunch.frame.copy()
    target_col = _find_target_column(df.columns)
    if target_col is None:
        raise ValueError("Unable to identify target column in OpenML bank-marketing dataset.")
    return df, target_col, "OpenML: bank-marketing (version=1)"


raw_df, target_col, provenance = load_bank_marketing_dataset()
print(f"Loaded from: {provenance}")
print(f"Shape: {raw_df.shape}")
print(f"Target column: {target_col}")
raw_df.head(3)

NameError: name 'Path' is not defined

## 2) Preprocessing and Leakage Control

Classical ML choice: for binary conversion prediction and contextual feature extraction, we combine:
- `SimpleImputer` for robust missing-value handling
- `OneHotEncoder` for categorical variables
- `StandardScaler` for numeric variables

Leakage handling: we explicitly remove `duration`, because call duration is not known before decision time.

In [ ]:
def normalize_target(series: pd.Series) -> pd.Series:
    values = series.astype(str).str.lower().str.strip()
    positive = {"yes", "1", "true", "y", "converted", "success"}
    return values.map(lambda v: 1 if v in positive else 0).astype(int)


df = raw_df.copy()
if "duration" in df.columns:
    df = df.drop(columns=["duration"])

y = normalize_target(df[target_col])
X = df.drop(columns=[target_col])

numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
    ],
    remainder="drop",
)

print(f"Features after leakage handling: {X.shape[1]}")
print(f"Numeric columns: {len(numeric_cols)}, Categorical columns: {len(categorical_cols)}")

## 3) Optional Classical ML Sanity Check

We train a simple logistic regression only as a sanity check that the preprocessed features carry conversion signal. This is not the bandit policy.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("logreg", LogisticRegression(max_iter=1000)),
])

clf.fit(X_train, y_train)
proba_test = clf.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, proba_test)
print(f"Validation ROC-AUC (sanity check): {auc:.4f}")

## 4) Build Synthetic Contextual Bandit Layer

To evaluate online policies reproducibly, we create a synthetic environment on top of real customer contexts:
- reduce transformed feature space to compact dense context vectors
- define an offer catalog with business margins
- define hidden conversion probabilities per offer (unknown to policies)
- generate logged interactions and delayed feedback

In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))


def build_context_matrix(
    X_data: pd.DataFrame,
    preproc: ColumnTransformer,
    n_components: int = 12,
) -> np.ndarray:
    X_sparse = preproc.fit_transform(X_data)
    max_comp = max(2, min(n_components, X_sparse.shape[1] - 1))
    reducer = TruncatedSVD(n_components=max_comp, random_state=42)
    X_dense = reducer.fit_transform(X_sparse)
    return X_dense


offer_catalog = pd.DataFrame(
    [
        {"arm": 0, "offer_name": "No Incentive", "margin": 1.0},
        {"arm": 1, "offer_name": "Fee Discount", "margin": 1.2},
        {"arm": 2, "offer_name": "Cashback", "margin": 1.4},
        {"arm": 3, "offer_name": "Premium Bundle", "margin": 1.8},
    ]
)

contexts = build_context_matrix(X, preprocessor, n_components=12)
n_rounds, context_dim = contexts.shape
n_arms = len(offer_catalog)
margins = offer_catalog["margin"].to_numpy()

true_weights = RNG.normal(loc=0.0, scale=0.35, size=(n_arms, context_dim))
true_bias = np.array([-0.6, -0.4, -0.3, -0.2])

all_logits = contexts @ true_weights.T + true_bias
all_probs = sigmoid(all_logits)
all_expected_rewards = all_probs * margins

print(f"Bandit rounds: {n_rounds}, context dim: {context_dim}, arms: {n_arms}")
offer_catalog

In [ ]:
def generate_logged_interactions(
    contexts_arr: np.ndarray,
    probs: np.ndarray,
    margins_arr: np.ndarray,
    epsilon: float = 0.35,
    seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n, k = probs.shape

    actions = np.zeros(n, dtype=int)
    chosen_prob = np.zeros(n, dtype=float)
    conversion = np.zeros(n, dtype=int)
    reward = np.zeros(n, dtype=float)
    delay_steps = np.zeros(n, dtype=int)

    for t in range(n):
        greedy_arm = int(np.argmax(probs[t] * margins_arr))
        if rng.random() < epsilon:
            a = int(rng.integers(0, k))
        else:
            a = greedy_arm

        p = probs[t, a]
        conv = int(rng.random() < p)
        r = conv * margins_arr[a]
        d = int(min(rng.geometric(0.35) - 1, 8))

        actions[t] = a
        chosen_prob[t] = p
        conversion[t] = conv
        reward[t] = r
        delay_steps[t] = d

    logged = pd.DataFrame({
        "round": np.arange(n),
        "action": actions,
        "true_p": chosen_prob,
        "conversion": conversion,
        "reward": reward,
        "delay_steps": delay_steps,
    })
    return logged


logged_df = generate_logged_interactions(contexts, all_probs, margins, epsilon=0.35, seed=42)
logged_df.head(10)

## 5) Policy Definitions

### Deterministic baseline
A fixed business rule: always send the same offer (`Premium Bundle`). This is simple and fully deterministic, but does not adapt to context.

### UCB intuition
Upper Confidence Bound adds optimism for uncertain arms. Arms with fewer observations get higher uncertainty bonuses, creating principled exploration.

### Thompson Sampling intuition
For Bernoulli conversion, each arm keeps a Beta posterior over conversion probability. At each round, sample one plausible conversion rate per arm and pick the arm with best sampled expected reward.

In [ ]:
@dataclass
class RoundOutcome:
    action: int
    conversion: int
    reward: float
    expected_reward: float
    optimal_expected_reward: float
    explored: int


class DeterministicBaselinePolicy:
    def __init__(self, fixed_arm: int, n_arms: int):
        self.fixed_arm = fixed_arm
        self.n_arms = n_arms

    def select_action(self, t: int) -> Tuple[int, int]:
        return self.fixed_arm, 0

    def update(self, action: int, conversion: int) -> None:
        return


class UCBPolicy:
    def __init__(self, n_arms: int, margins_arr: np.ndarray, alpha: float = 1.0):
        self.n_arms = n_arms
        self.margins = margins_arr
        self.alpha = alpha
        self.successes = np.zeros(n_arms, dtype=float)
        self.failures = np.zeros(n_arms, dtype=float)

    @property
    def counts(self) -> np.ndarray:
        return self.successes + self.failures

    def select_action(self, t: int) -> Tuple[int, int]:
        counts = self.counts
        total = np.maximum(1.0, counts.sum())
        mean = (self.successes + 1.0) / (counts + 2.0)
        bonus = self.alpha * np.sqrt(np.log(total + 1.0) / (counts + 1.0))
        ucb_score = (mean + bonus) * self.margins

        greedy = int(np.argmax(mean * self.margins))
        action = int(np.argmax(ucb_score))
        explored = int(action != greedy)
        return action, explored

    def update(self, action: int, conversion: int) -> None:
        if conversion:
            self.successes[action] += 1.0
        else:
            self.failures[action] += 1.0


class ThompsonSamplingPolicy:
    def __init__(self, n_arms: int, margins_arr: np.ndarray):
        self.n_arms = n_arms
        self.margins = margins_arr
        self.alpha = np.ones(n_arms, dtype=float)
        self.beta = np.ones(n_arms, dtype=float)

    def select_action(self, t: int) -> Tuple[int, int]:
        sampled_theta = RNG.beta(self.alpha, self.beta)
        sampled_reward = sampled_theta * self.margins

        post_mean = self.alpha / (self.alpha + self.beta)
        greedy = int(np.argmax(post_mean * self.margins))
        action = int(np.argmax(sampled_reward))
        explored = int(action != greedy)
        return action, explored

    def update(self, action: int, conversion: int) -> None:
        self.alpha[action] += conversion
        self.beta[action] += 1 - conversion

## 6) Simulation with Delayed Rewards

We simulate online decisions over customer contexts.

Delayed reward handling:
- each impression gets a delay in rounds before feedback is observable
- policy updates happen only when delayed feedback matures
- metrics are tracked by decision time

In [ ]:
def run_policy_simulation(
    policy_name: str,
    policy,
    probs: np.ndarray,
    margins_arr: np.ndarray,
    max_delay: int = 8,
    seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    n, _ = probs.shape

    pending: Dict[int, List[Tuple[int, int]]] = {}
    rows: List[RoundOutcome] = []

    for t in range(n):
        matured = pending.pop(t, [])
        for act, conv in matured:
            policy.update(act, conv)

        action, explored = policy.select_action(t)
        p = probs[t, action]
        conversion = int(rng.random() < p)
        reward = conversion * margins_arr[action]

        delay = int(min(rng.geometric(0.35) - 1, max_delay))
        update_t = t + delay
        pending.setdefault(update_t, []).append((action, conversion))

        expected_reward = p * margins_arr[action]
        optimal_expected = float(np.max(probs[t] * margins_arr))

        rows.append(
            RoundOutcome(
                action=action,
                conversion=conversion,
                reward=reward,
                expected_reward=expected_reward,
                optimal_expected_reward=optimal_expected,
                explored=explored,
            )
        )

    result = pd.DataFrame([r.__dict__ for r in rows])
    result["policy"] = policy_name
    result["round"] = np.arange(len(result))
    result["instant_regret"] = result["optimal_expected_reward"] - result["expected_reward"]
    result["cum_reward"] = result["reward"].cumsum()
    result["cum_regret"] = result["instant_regret"].cumsum()
    result["cum_conversions"] = result["conversion"].cumsum()
    result["cum_conversion_rate"] = result["cum_conversions"] / (result["round"] + 1)
    result["cum_exploration_share"] = result["explored"].cumsum() / (result["round"] + 1)
    return result


baseline_policy = DeterministicBaselinePolicy(fixed_arm=3, n_arms=n_arms)
ucb_policy = UCBPolicy(n_arms=n_arms, margins_arr=margins, alpha=1.2)
ts_policy = ThompsonSamplingPolicy(n_arms=n_arms, margins_arr=margins)

sim_baseline = run_policy_simulation("Deterministic Baseline", baseline_policy, all_probs, margins, seed=100)
sim_ucb = run_policy_simulation("UCB", ucb_policy, all_probs, margins, seed=100)
sim_ts = run_policy_simulation("Thompson Sampling", ts_policy, all_probs, margins, seed=100)

sim_all = pd.concat([sim_baseline, sim_ucb, sim_ts], ignore_index=True)
sim_all.head()

## 7) Aggregate Metrics

In [ ]:
summary = (
    sim_all.groupby("policy", as_index=False)
    .agg(
        total_reward=("reward", "sum"),
        cumulative_regret=("instant_regret", "sum"),
        conversion_rate=("conversion", "mean"),
        exploration_share=("explored", "mean"),
    )
    .sort_values(by="total_reward", ascending=False)
)
summary

## 8) Two-Way Foundry Agent Integration

This step demonstrates the agentic loop on the workload:

1. The notebook sends only aggregate policy evidence to an Azure AI Foundry agent.
2. The agent validates and enriches the multi-armed strategy.
3. The notebook accepts only safe bounded recommendations and reruns a candidate policy.

If Foundry SDK configuration is not available, the same contract uses a deterministic local fallback so the pipeline remains reproducible.

In [ ]:
import json
import os
import sys

candidate_src_paths = [
    Path.cwd() / "src",
    Path.cwd().parent / "aml-foundry-integration" / "src",
    Path.cwd().parent.parent / "aml-foundry-integration" / "src",
]
for package_src in candidate_src_paths:
    if (package_src / "aml_bandits" / "foundry_bridge.py").exists():
        if str(package_src) not in sys.path:
            sys.path.insert(0, str(package_src))
        break

from aml_bandits.foundry_bridge import (
    build_aggregate_metrics,
    to_agent_payload,
    validate_and_enrich_strategy,
)

aggregate_evidence = build_aggregate_metrics(summary, current_ucb_alpha=1.2)
agent_payload = to_agent_payload(aggregate_evidence)

print("Payload privacy mode:", agent_payload["privacy"])
print("Allowed recommendations:", agent_payload["allowed_recommendations"])
print("Foundry endpoint configured:", bool(os.getenv("FOUNDRY_PROJECT_ENDPOINT") or os.getenv("AZURE_AI_PROJECT_ENDPOINT")))
print("Foundry agent configured:", bool(os.getenv("FOUNDRY_AGENT_ID") or os.getenv("AZURE_AI_AGENT_ID") or os.getenv("FOUNDRY_AGENT_NAME") or os.getenv("AZURE_AI_AGENT_NAME")))

agent_recommendation = validate_and_enrich_strategy(aggregate_evidence)

agent_review = pd.DataFrame([
    {
        "source": agent_recommendation.source,
        "validation": agent_recommendation.validation,
        "accepted": agent_recommendation.accepted,
        "recommended_ucb_alpha": agent_recommendation.recommended_ucb_alpha,
        "bounded_ucb_alpha": agent_recommendation.bounded_ucb_alpha,
        "rationale": agent_recommendation.rationale,
        "warnings": "; ".join(agent_recommendation.warnings),
    }
])
agent_review

In [ ]:
if agent_recommendation.accepted and agent_recommendation.bounded_ucb_alpha is not None:
    agent_ucb_policy = UCBPolicy(
        n_arms=n_arms,
        margins_arr=margins,
        alpha=agent_recommendation.bounded_ucb_alpha,
    )
    sim_ucb_agent = run_policy_simulation(
        "UCB (Agent Recommended)",
        agent_ucb_policy,
        all_probs,
        margins,
        seed=101,
    )
    sim_all = pd.concat([sim_all, sim_ucb_agent], ignore_index=True)
    summary = (
        sim_all.groupby("policy", as_index=False)
        .agg(
            total_reward=("reward", "sum"),
            cumulative_regret=("instant_regret", "sum"),
            conversion_rate=("conversion", "mean"),
            exploration_share=("explored", "mean"),
        )
        .sort_values(by="total_reward", ascending=False)
    )
    print(
        "Applied agent recommendation: "
        f"UCB alpha={agent_recommendation.bounded_ucb_alpha:.3f}"
    )
else:
    print("No safe agent recommendation was applied; baseline policy set remains unchanged.")

summary

## 9) Visual Comparison

Simple diagnostics:
- cumulative reward (higher is better)
- cumulative regret (lower is better)
- running conversion rate
- running exploration share

If the Foundry bridge accepted a safe recommendation, the `UCB (Agent Recommended)` candidate appears alongside the baseline policies.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for policy_name, grp in sim_all.groupby("policy"):
    axes[0, 0].plot(grp["round"], grp["cum_reward"], label=policy_name)
    axes[0, 1].plot(grp["round"], grp["cum_regret"], label=policy_name)
    axes[1, 0].plot(grp["round"], grp["cum_conversion_rate"], label=policy_name)
    axes[1, 1].plot(grp["round"], grp["cum_exploration_share"], label=policy_name)

axes[0, 0].set_title("Cumulative Reward")
axes[0, 1].set_title("Cumulative Regret")
axes[1, 0].set_title("Running Conversion Rate")
axes[1, 1].set_title("Running Exploration Share")

for ax in axes.ravel():
    ax.set_xlabel("Round")
    ax.grid(alpha=0.25)

axes[0, 0].set_ylabel("Reward")
axes[0, 1].set_ylabel("Regret")
axes[1, 0].set_ylabel("Conversion Rate")
axes[1, 1].set_ylabel("Exploration Share")

axes[0, 0].legend(loc="best")
axes[0, 1].legend(loc="best")
axes[1, 0].legend(loc="best")
axes[1, 1].legend(loc="best")

plt.tight_layout()
plt.show()

## 10) Findings and Limitations

### Findings
- Adaptive policies (UCB and Thompson Sampling) typically improve reward over a fixed deterministic baseline.
- UCB tends to be systematic in exploration through uncertainty bonuses.
- Thompson Sampling often balances exploration and exploitation naturally via posterior sampling.
- Delayed rewards slow learning because policy updates do not happen immediately after each action.
- The Foundry agent bridge demonstrates a two-way integration pattern: aggregate evidence leaves the pipeline, bounded strategy guidance returns, and the notebook reruns a candidate policy.

### Limitations
- Reward model is synthetic (contextual but simulated), not direct production uplift.
- Only Bernoulli conversion is modeled; no long-term customer value or churn effects.
- No propensity correction or off-policy evaluation estimator is applied in this baseline notebook.
- Offer economics are simplified to scalar margins; real campaigns may need budget and fairness constraints.
- Agent guidance is intentionally constrained to aggregate-only evidence and bounded UCB alpha changes; broader strategy changes should pass separate review gates.

### Next Steps for Datathon
- Add contextual bandits with linear models (LinUCB / contextual Thompson).
- Add policy constraints (max discount budget, segment-level caps).
- Evaluate with offline estimators (IPS/DR) before online deployment.
- Register the Foundry agent response and accepted recommendation as experiment artifacts in Azure ML.